In [0]:
#Libraries management
from pyspark import pipelines as pl
from pyspark.sql.functions import *
from pyspark.sql.types import *

volume_path="/Volumes/workspace/damg7370/datastore/Schema_Drift_Rescue/customer_*.json" 


In [0]:
#bronze layer table: cust_bronze_sd
pl.create_streaming_table("demo_cust_bronze_sd")

# Ingest the raw data into the bronze table using append flow
@pl.append_flow(
  target = "demo_cust_bronze_sd", #object name
  name = "demo_cust_bronze_sd_ingest_flow" #flow name
)
def demo_cust_bronze_sd_ingest_flow():
  df = (
      spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.inferColumnTypes", "true") #auto scan schema 
          #.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns") # schema customer_data_1.json is different than customer_data_2.json so it fails with  [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_RECORD_WITH_FILE_PATH] excetion and stops processing
          .option("cloudFiles.schemaEvolutionMode", "rescue")
          .load(f"{volume_path}")
  )
  return df.withColumn("ingestion_datetime", current_timestamp())\
           .withColumn("source_filename", col("_metadata.file_path")) 


In [0]:
# Function to handle DATATYPE changes
# Logic to process the fields if data type changes. There are many ways it can be handled
#    (*) Without Overwrite of data in silver layer
#        Create additional table every time a colmn datatype changes and then create view on top of it as UNION to all new tables
#            PROS: Technically we are not overwriting the data hence no reloading
#            CONS: New table will created every time datatype changes 
#
#    (*) Overwrite data in both Bronze and Silver layers
#        I am not sure how this works for streams. I have not done much exploration in this method. Hope it works
#        Here aswell we use _rescued_data column to check qualit expectation for schema update
#            PROS: No need to reload bronze layer table as _rescued_data has all desired changed and can be used to process
#            CONS: Less code changes because _rescued_data doesnt need any additional logic to hendle. 
#                  However all raw data to be stored at begining. Reload of both tables
#
#    (*) Merge and overwrite data in silver layer (below function example does the same implementation)
#            PROS: No need to reload bronze layer table as _rescued_data has all desired changed and can be used to process
#            CONS: Table in silver need to be completely reloaded
# NOTE: The above options technically doesnt handle column renames. We need to write additional logic to handle column renames
#       I would say we could follow the views logic to load renamed column as new field and then in view drop old column and use new renamed column
#       However, we need to merge the data in silver layer and hence we need to reload the silver layer table

def process__rescue_data_datatype_change(df, target_schema: StructType):
    #Parse the _rescued_data json to a MAP (Key,Value) type and store in _rescued_data_modified column
    df = df.withColumn("_rescued_data_modified", from_json(col("_rescued_data"), MapType(StringType(), StringType())))
    
    for field in target_schema.fields:
        data_type = field.dataType
        column_name = field.name

        # Check if "_rescue_data" is not null and if the key exists
        # pyspark.sql.functions.map_contains_key function in PySpark is used to check if a specified key exists within a MapType column in a DataFrame. returns T/F
        key_condition = expr(f"_rescued_data_modified IS NOT NULL AND map_contains_key(_rescued_data_modified, '{column_name}')")
        
        # Extract the rescued value for this column, if it exists, and cast it to the target data type
        rescued_value = when(key_condition, col("_rescued_data_modified").getItem(column_name).cast(data_type)).otherwise(col(column_name).cast(data_type))
        
        # Update the DataFrame with the merged column
        df = df.withColumn(column_name, rescued_value)
        df = df.withColumn(column_name, col(column_name).cast(data_type))
        
    df = df.drop('_rescued_data_modified')

    # Setting the _rescued_data to null after processing since we use the column to check qualit expectation for schema update
    df = df.withColumn('_rescued_data', lit(None).cast(StringType()))
    return df


In [0]:
# Function to handle adding NEW FIELDS 
def process__rescue_data_new_fields(df):
    
    # Add all fields from _rescued_data to key map
    df = df.withColumn(
        "_rescued_data_json_to_map", 
        from_json(
            col("_rescued_data"), 
            MapType(StringType(), StringType())
        )
    )

    # Define all possible fields that might appear in _rescued_data across all files
    # This is the most reliable approach for streaming
    possible_rescued_fields = ["Age", "Gender", "LoyaltyStatus", "CreditScore", "SignupDate"]
    
    # Get existing columns in the dataframe
    existing_columns = set(df.columns)
    
    # Add new columns for each possible field if it doesn't already exist
    for key in possible_rescued_fields:
        if key not in existing_columns:
            # Conditionally add the column only if it exists in the rescued data map
            df = df.withColumn(
                key,
                when(
                    col("_rescued_data_json_to_map").isNotNull(),
                    col("_rescued_data_json_to_map").getItem(key)
                ).otherwise(lit(None))
            )
    
    # Clean up temporary column
    df = df.drop("_rescued_data_json_to_map")
    
    return df



In [0]:

# #plain implementation without processing _rescue_data field. Use this when you upload customer_data_1.json
# # -----------------------------------------------------------------------------------------------------
# pl.create_streaming_table(
#   name = "demo_cust_silver_sd",
#   expect_all_or_drop = {"no_rescued_data": "_rescued_data IS NULL","valid_id": "CustomerID IS NOT NULL"}
#   )
# @pl.append_flow(
#   target = "demo_cust_silver_sd",
#   name = "demo_cust_silver_sd_clean_flow"
# )
# def demo_cust_silver_sd_clean_flow():
#   return (
#       spark.readStream.table("demo_cust_bronze_sd")
#   )

In [0]:
# -----------------------------------------------------------------------------------------------------
# uncomment this code before uploading customer_data_2.json. Then upload the file and run the pipeline
# -----------------------------------------------------------------------------------------------------
# we know that when we process customer_data_2.json file there are new fields in schema to be added and at same time we are planing for datatype chage
# for an already existing field that came with customer_data_1.json file. Since there is a datatype change for existing field so we need to perform
# full refresh (Run pipeline with full table refresh)
updated_datatypes = StructType([
  # define the column signuoDate as DATE type and also make it nullable (Make 3rd argument False if you want to make it non nullable)
  StructField("signupDate", DateType(), True) ,
  StructField("CreditScore", IntegerType(), True)
])

pl.create_streaming_table(
  name = "demo_cust_silver_sd",
  expect_all_or_drop = {"no_rescued_data": "_rescued_data IS NULL","valid_id": "CustomerID IS NOT NULL"}
)

@pl.append_flow(
  target = "demo_cust_silver_sd",
  name = "demo_cust_silver_sd_clean_flow"
)
def demo_cust_silver_sd_clean_flow():
  df = (
    spark.readStream.table("demo_cust_bronze_sd")
  )
  df = process__rescue_data_new_fields(df)
  df = process__rescue_data_datatype_change(df, updated_datatypes)
  return df

